In [ ]:
#------------- Initialize libraries and files
import pandas as pd
import numpy as np
import statsmodels.api as sm
data=pd.read_excel(os.path.join(os.getcwd(),"auctionScraper-2.xlsm"),sheet_name='Sheet3')

In [102]:
#------------Define main problems thata may be in the text of descriptions and create those variables
vic=['ريورك','ايربگ','بد كار','چراغ چك','پروژه اي','دنده','تصادفي','تصادف شديد','اي بي اس','صداي','شماره گذاري','سقف','رنگي','گيربكس','جلوبند','آچارخوردگي','مستعمل','موتور','مغاير','لاستيك','باطري','شكستگي','اوراق']
for col in vic:
    data[col]=data['ايرادات خودرو'].str.contains(col)

In [103]:
#=============== Main statistical modeling section ================
#data.drop(axis=0,labels=data[data.marketValue.isna()==True].index,inplace=True)
data=data.sample(frac=1)
dataX=data
test=data[data.marketValue==0]
#dataX=data[data.marketValue>0]
#exog=dataX[np.append(vic,['قیمت'])]
exog=dataX[np.append(vic,['کیلومتر کارکرد'])]
#exog=dataX[vic]

exog=sm.add_constant(exog)
endog=dataX['winner']

#exog[vic]=exog[vic].astype('float')
exog=exog.join((dataX.year))
exog=exog.astype('float')
exog=exog.join(pd.get_dummies(dataX.carBrand))
exog[exog['کیلومتر کارکرد']==999999]['کیلومتر کارکرد']=np.nan
exog['کیلومتر کارکرد']=exog['کیلومتر کارکرد'].fillna('backfill')
#exog=exog.join(pd.get_dummies(dataX.difference))
#exog=exog.join(pd.get_dummies(dataX['وضعيت خودرو']),rsuffix='-status')
#exog=exog.join(pd.get_dummies(dataX.year))
idx=endog[endog.notna()==True].index.values
idxTest=endog[endog.notna()==False].index.values
mod = sm.OLS(endog.loc[idx], exog.loc[idx])

res = mod.fit()

print(res.summary())

                            OLS Regression Results                            
Dep. Variable:                 winner   R-squared:                       0.879
Model:                            OLS   Adj. R-squared:                  0.873
Method:                 Least Squares   F-statistic:                     133.8
Date:                Wed, 01 Feb 2023   Prob (F-statistic):               0.00
Time:                        13:39:43   Log-Likelihood:                -25145.
No. Observations:                1164   AIC:                         5.041e+04
Df Residuals:                    1103   BIC:                         5.072e+04
Df Model:                          60                                         
Covariance Type:            nonrobust                                         
                     coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------
const          -2.044e+11   3.36e+10     -6.

C:\Users\2740554486\AppData\Roaming\Python\Python37\site-packages\ipykernel_launcher.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy


In [97]:
#------- QC with model ------------
result=data.loc[idxTest]
result['predict']=res.predict(exog.loc[idxTest])
result

,نام خودرو,شماره شاسی,کد خودرو,رنگ خودرو,نوع سوخت,وضعيت خودرو,بيمه ثالث,کیلومتر کارکرد,year,شماره موتور,...,لاستيك,باطري,شكستگي,تصادف شديد,اوراق,ايربگ,بد كار,لرزش,دنده,predict
1587,دنا كلاس 2 سفيد دو پوششه,HE461600,701,سفيد دو پوششه,بنزين,خودرو اوراقي,NaN,999999,1396,147H0279448,...,True,True,True,False,True,False,False,False,False,2.710840e+09
1472,پژو 207i اتوماتيک با موتور TU5P کلاس 3 سفيد دو...,NJ240536,606,سفيد دو پوششه,بنزين,ايراد رنگي بدنه,NaN,63,1401,185B0000929,...,False,False,False,False,False,False,False,False,False,5.044691e+09
1346,تارا دستي کلاس 1 سفيد دو پوششه,MD916726,322,سفيد دو پوششه,بنزين,ماندگاري بالا و افت مدل,NaN,38,1400,187B0009220,...,False,True,False,False,False,False,False,False,False,3.853425e+09
1467,پژو 207i دستي کلاس 10 مشكي متاليك,MJ538270,600,مشكي متاليك,بنزين,ماندگاري بالا و افت مدل,NaN,67,1400,178B0074388,...,False,False,False,False,False,False,False,False,False,4.298994e+09
1584,دنا+ (پلاس) كلاس 1 سفيد دو پوششه,FE060002,703,سفيد دو پوششه,بنزين,خودرو اوراقي,NaN,999999,1394,147H0142252,...,True,True,True,False,True,False,False,False,True,1.928109e+09
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1674,سورن EF7 بنزيني کلاس 19 سفيد دو پوششه,NF159364,575,سفيد دو پوششه,بنزين,تصادفي,NaN,28,1401,147H0639185,...,False,True,True,False,False,False,False,False,False,3.486549e+09
1245,دنا پلاس توربو شارژ کلاس 6 خاكستري متاليك,NE490590,704,خاكستري متاليك,بنزين,ايراد رنگي بدنه,NaN,20,1401,153H0055268,...,False,True,False,False,False,False,False,False,False,4.782301e+09
1369,تارا دستي کلاس 1 قهوه اي كادرو,MD912440,322,قهوه اي كادرو,بنزين,ماندگاري بالا و افت مدل,NaN,47,1400,187B0003057,...,False,True,False,False,False,False,False,False,False,3.725893e+09
1343,تارا دستي کلاس 1 خاكستري متاليك,ND934745,322,خاكستري متاليك,بنزين,خودروي كاردكسي- قابل شماره گذاري توسط برنده مز...,NaN,39,1401,187B0015428,...,False,False,True,False,False,False,False,False,False,4.102798e+09


In [101]:
#---------Save modleing info
cols=['predict','قیمت','marketValue','diff','rev','نام خودرو', 'شماره شاسی', 'کد خودرو', 'رنگ خودرو', 'نوع سوخت',
       'وضعيت خودرو', 'بيمه ثالث', 'کیلومتر کارکرد', 'year', 'شماره موتور',
       'ايرادات خودرو', 'شماره پلاك انتظامي', 'carBrand', 'کد کلاس', 'carType',
       'کد کلاس: ', 'شماره بدنه:', 'شماره جایگاه:', 'reserve', 'قیمت',
       'winner', 'percent higher', 'id', 'marketValue', 'difference', 'ريورك',
       'ايربك', 'استارت', 'چراغ چك', 'تصادفي', 'اي بي اس', 'صداي',
       'شماره گذاري', 'سقف', 'رنگي', 'گيربكس', 'جلوبند', 'آچارخوردگي',
       'مستعمل', 'موتور', 'مغاير', 'لاستيك', 'باطري', 'شكستگي', 'تصادف شديد',
       'اوراق', 'ايربگ', 'بد كار', 'لرزش', 'دنده']
result[cols].to_excel('res2.xlsx')

# check with real value of a non-damaged car
b=pd.DataFrame()
b['pre']=res.predict(exog.loc[1001:exog.shape[0]])
b['real']=endog[1001:endog.shape[0]]
b['diff']=b['pre']-b['real']
c=b.join(exog.iloc[1001:exog.shape[0]])
c['diff2']=c.real-c['قیمت']

In [ ]:
# ========= Save the model for futher loadings
b.save('resModel.pickle')

In [ ]:
#============== predict newly arrived car prices to bid
from statsmodels.iolib.smpickle import load_pickle
new_results = load_pickle('resModel.pickle')
new_results.predict(exog.iloc[1001:exog.shape[0]])
c=b.join(exog.iloc[151:exog.shape[0]])
testY=test['winner']

testX=test[np.append(vic,['کیلومتر کارکرد'])]
testX=sm.add_constant(testX)


testX=testX.astype('float')
testX=testX.join(pd.get_dummies(test.carBrand))
testX=testX.join(pd.get_dummies(test.year))
testX.marketValue=np.nan
b=pd.DataFrame()
b['preY']=res.predict(testX)
b['real']=testY

In [ ]:
# ==== QC model with added variables to be used in the model
vic.append('reserve','marketValue')

exog=data[np.append(vic,['reserve','marketValue'])]
exog=sm.add_constant(exog)
endog=data['winner']

exog[vic]=exog[vic].astype('float')
exog.join(pd.get_dummies(data.carBrand))
exog.join(pd.get_dummies(data.year))

mod = sm.OLS(endog, exog)

res = mod.fit()

print(res.summary())